# 02 Modeling - Airbnb Price Prediction

This notebook trains and evaluates regression models using the processed dataset created by the EDA workflow.

In [ ]:
import sys
from pathlib import Path
import importlib

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV

project_dir = Path('/Users/andreyvargassolis/Desktop/Data_Science/AirBnB_PricePerCity')
if str(project_dir / 'src') not in sys.path:
    sys.path.append(str(project_dir / 'src'))

import evaluate_model
import preprocessing
import train_model

importlib.reload(evaluate_model)
importlib.reload(preprocessing)
importlib.reload(train_model)

from evaluate_model import evaluate_models, regression_metrics, select_cv_finalists, select_final_model
from preprocessing import (
    MODEL_FEATURES,
    PROCESSED_DATA_PATH,
    TARGET,
    build_model_dataframe,
    get_categorical_columns,
    get_categorical_levels,
    load_raw_data,
    one_hot_encode,
    save_processed_data,
)
from train_model import (
    BEST_MODEL_PATH,
    get_models,
    plot_feature_importance,
    plot_price_distribution,
    split_features_target,
)

sns.set_theme(style='whitegrid')

In [ ]:
raw_df = load_raw_data()
model_df, model_input_df, metadata = build_model_dataframe(raw_df)
save_processed_data(model_df)

print(f'Model dataframe shape: {model_df.shape}')
display(model_df.head())

In [ ]:
X_train, X_cv, X_test, y_train, y_cv, y_test = split_features_target(model_df)

print(f'X_train shape: {X_train.shape}')
print(f'X_cross_validation shape: {X_cv.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Train percentage: {len(X_train) / len(model_df) * 100:.1f}%')
print(f'Cross-validation percentage: {len(X_cv) / len(model_df) * 100:.1f}%')
print(f'Test percentage: {len(X_test) / len(model_df) * 100:.1f}%')

## Baseline And Tree Models

In [ ]:
models = get_models()
model_results_df = evaluate_models(models, X_train, X_cv, X_test, y_train, y_cv, y_test)

cv_ranking_df = model_results_df.sort_values('CV_RMSE').reset_index(drop=True)
cv_finalists_df = select_cv_finalists(model_results_df, n_finalists=2)

print('Models ranked by cross-validation RMSE:')
display(cv_ranking_df[['model', 'Train_RMSE', 'CV_MAE', 'CV_RMSE', 'CV_R2', 'CV_MAPE_%', 'CV_Overfit_Gap_RMSE']])

print('Top 2 models selected by cross-validation:')
display(cv_finalists_df[['model', 'CV_RMSE', 'CV_R2', 'CV_Overfit_Gap_RMSE']])

In [ ]:
best_model_name, final_test_results_df = select_final_model(model_results_df, n_finalists=2)
best_model = models[best_model_name]

print('Only the top 2 cross-validation models are evaluated for final selection on the test set.')
print('Final_Selection_Score = Test_R2 - max(Test_Overfit_Gap_RMSE, 0) / Test_RMSE')
print(f'Best final model: {best_model_name}')

display(final_test_results_df[[
    'model',
    'Test_MAE',
    'Test_RMSE',
    'Test_R2',
    'Test_MAPE_%',
    'Test_Overfit_Gap_RMSE',
    'Final_Selection_Score',
]])

## Random Forest Hyperparameter Tuning

In [ ]:
# Optional tuning. This cell can take a few minutes.
# The search space focuses on anti-overfitting parameters.
rf_param_grid = {
    'n_estimators': [250, 300, 500],
    'max_depth': [18, 22, 28],
    'min_samples_split': [8, 10, 16],
    'min_samples_leaf': [4, 5, 8],
    'max_features': [0.7, 0.8],
    'bootstrap': [True],
    'max_samples': [0.7, 0.75, 0.85],
}

rf_random_search = RandomizedSearchCV(
    estimator=models['Random Forest Regressor'],
    param_distributions=rf_param_grid,
    n_iter=12,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1,
)

rf_random_search.fit(X_train, y_train)

tuned_rf = rf_random_search.best_estimator_
y_train_pred = tuned_rf.predict(X_train)
y_cv_pred = tuned_rf.predict(X_cv)
y_test_pred = tuned_rf.predict(X_test)

cv_metrics = regression_metrics(
    y_cv,
    y_cv_pred,
    y_train=y_train,
    y_train_pred=y_train_pred,
)
test_metrics = regression_metrics(
    y_test,
    y_test_pred,
    y_train=y_train,
    y_train_pred=y_train_pred,
)

tuned_rf_metrics = {
    'model': 'Tuned Random Forest Regressor',
    'Train_RMSE': cv_metrics['Train_RMSE'],
    'CV_MAE': cv_metrics['MAE'],
    'CV_RMSE': cv_metrics['RMSE'],
    'CV_R2': cv_metrics['R2'],
    'CV_MAPE_%': cv_metrics['MAPE_%'],
    'Test_MAE': test_metrics['MAE'],
    'Test_RMSE': test_metrics['RMSE'],
    'Test_R2': test_metrics['R2'],
    'Test_MAPE_%': test_metrics['MAPE_%'],
    'CV_Overfit_Gap_RMSE': cv_metrics['Overfit_Gap_RMSE'],
    'Test_Overfit_Gap_RMSE': test_metrics['Overfit_Gap_RMSE'],
}
tuned_rf_metrics['Generalization_Score'] = (
    tuned_rf_metrics['CV_RMSE']
    + 0.35 * max(tuned_rf_metrics['CV_Overfit_Gap_RMSE'], 0)
)

tuning_comparison = pd.concat(
    [
        model_results_df[model_results_df['model'] == 'Random Forest Regressor'],
        pd.DataFrame([tuned_rf_metrics]),
    ],
    ignore_index=True,
).sort_values('Generalization_Score')

print('Best Random Forest parameters:')
print(rf_random_search.best_params_)
print(f"Best cross-validation RMSE from RandomizedSearchCV: {-rf_random_search.best_score_:.2f}")
display(tuning_comparison)

## Save Best Model And Images

In [ ]:
model_artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'feature_columns': X_train.columns.tolist(),
    'target': TARGET,
    'metrics': model_results_df,
    'finalist_metrics': final_test_results_df,
    'selection_rule': 'Top 2 by CV_RMSE, final choice by Test_R2 minus relative overfitting gap.',
    'split_strategy': {
        'train': 0.6,
        'cross_validation': 0.2,
        'test': 0.2,
        'random_state': 42,
    },
    'metadata': metadata,
}

BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_artifact, BEST_MODEL_PATH)

price_image_path = plot_price_distribution(raw_df)
importance_image_path = plot_feature_importance(best_model, X_train.columns, X=X_test, y=y_test)

print(f'Saved model to: {BEST_MODEL_PATH}')
print(f'Saved price distribution image to: {price_image_path}')
print(f'Saved feature importance image to: {importance_image_path}')

## Model Comparison Plots

In [ ]:
finalist_plot_df = final_test_results_df.copy()

error_metrics = finalist_plot_df.melt(
    id_vars='model',
    value_vars=['Test_MAE', 'Test_RMSE'],
    var_name='metric',
    value_name='value',
)

plt.figure(figsize=(9, 5))
sns.barplot(data=error_metrics, x='model', y='value', hue='metric')
plt.title('Finalists: MAE and RMSE on test set')
plt.xlabel('Model')
plt.ylabel('Error')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=finalist_plot_df, x='model', y='Test_R2', color='seagreen')
plt.title('Finalists: R2 on test set')
plt.xlabel('Model')
plt.ylabel('Test R2')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=finalist_plot_df, x='model', y='Test_Overfit_Gap_RMSE', color='indianred')
plt.title('Finalists: overfitting gap on test set')
plt.xlabel('Model')
plt.ylabel('Test RMSE - Train RMSE')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=finalist_plot_df, x='model', y='Final_Selection_Score', color='teal')
plt.title('Finalists: final selection score')
plt.xlabel('Model')
plt.ylabel('Higher is better')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Predict New Airbnb Prices

In [ ]:
categorical_columns = metadata['categorical_columns']
categorical_levels = metadata['categorical_levels']
feature_columns = model_artifact['feature_columns']


def predict_airbnb_price(airbnb_features):
    """Predict price_total from selected pre-listing Airbnb variables."""
    missing_columns = [feature for feature in MODEL_FEATURES if feature not in airbnb_features]

    if missing_columns:
        raise ValueError(f'Missing variables for prediction: {missing_columns}')

    airbnb_df = pd.DataFrame([{feature: airbnb_features[feature] for feature in MODEL_FEATURES}])

    unknown_categories = {}
    for column, levels in categorical_levels.items():
        value = airbnb_df.loc[0, column]
        if value not in levels:
            if column == 'district' and 'Other' in levels:
                value = 'Other'
                airbnb_df.loc[0, column] = value
            else:
                unknown_categories[column] = value
        airbnb_df[column] = pd.Categorical(airbnb_df[column], categories=levels)

    if unknown_categories:
        raise ValueError(f'Unknown categorical values: {unknown_categories}')

    airbnb_encoded = one_hot_encode(
        airbnb_df,
        categorical_columns=categorical_columns,
        categorical_levels=categorical_levels,
        target=None if TARGET not in airbnb_df.columns else TARGET,
    )
    airbnb_encoded = airbnb_encoded.reindex(columns=feature_columns, fill_value=0)

    return best_model.predict(airbnb_encoded)[0]

print(f'Prediction function is using: {best_model_name}')

In [ ]:
example_airbnbs = [
    {
        'room_type': 'Entire home/apt',
        'max_guests': 4.0,
        'num_bedrooms': 2,
        'distance_city_center': 1.2,
        'distance_metro': 0.4,
        'attraction_index_norm': 22.0,
        'restaurant_index_norm': 35.0,
        'proximity_index': 1.8,
        'city': 'Amsterdam',
        'district': 'Gemeente Amsterdam',
        'day_type': 'weekend',
    },
    {
        'room_type': 'Private room',
        'max_guests': 2.0,
        'num_bedrooms': 1,
        'distance_city_center': 4.5,
        'distance_metro': 1.8,
        'attraction_index_norm': 8.0,
        'restaurant_index_norm': 12.0,
        'proximity_index': 0.7,
        'city': 'Paris',
        'district': 'Paris',
        'day_type': 'weekday',
    },
    {
        'room_type': 'Shared room',
        'max_guests': 1.0,
        'num_bedrooms': 1,
        'distance_city_center': 7.0,
        'distance_metro': 2.8,
        'attraction_index_norm': 4.0,
        'restaurant_index_norm': 6.0,
        'proximity_index': 0.35,
        'city': 'Berlin',
        'district': 'Berlin',
        'day_type': 'weekday',
    },
]

for index, airbnb in enumerate(example_airbnbs, start=1):
    predicted_price = predict_airbnb_price(airbnb)
    print(f'Example {index} predicted price_total: {predicted_price:.2f}')

## Final Interpretation

In [ ]:
best_row = final_test_results_df.loc[0]
second_row = final_test_results_df.loc[1]

print(f"Best final model: {best_row['model']}")
print(f"Test RMSE: {best_row['Test_RMSE']:.2f}")
print(f"Test MAE: {best_row['Test_MAE']:.2f}")
print(f"Test R2: {best_row['Test_R2']:.3f}")
print(f"Test MAPE: {best_row['Test_MAPE_%']:.2f}%")
print(f"Test overfitting gap RMSE: {best_row['Test_Overfit_Gap_RMSE']:.2f}")
print(f"Final selection score: {best_row['Final_Selection_Score']:.3f}")

print()
print('Interpretation:')
print('- First, all models are compared using cross-validation. The two lowest CV_RMSE models become finalists.')
print('- Then only those two finalists are compared on the test set.')
print('- The final model is selected by balancing explained variance and overfitting: high Test_R2 is good, while a large Test_Overfit_Gap_RMSE is penalized.')
print('- MAE and RMSE measure prediction error in price_total units. RMSE is usually higher because it penalizes large errors more strongly.')
print('- R2 measures how much of the price variation is explained by the model. Higher R2 means the model explains prices better.')
print('- The overfitting gap compares test error with train error. A smaller gap means the model generalizes more honestly and is less likely to be memorizing training examples.')
print(f"- Compared with {second_row['model']}, {best_row['model']} is preferred because it has the better final balance between Test_R2 and overfitting gap.")